In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor

# Import Bayesian Search from scikit-optimize
from skopt import BayesSearchCV
from skopt.space import Real, Integer






In [6]:
# ==========================================
# 1. SELECT PARAMETER (FULLY DYNAMIC)
# Options: 'iddq_uA' | 'leakage_current_nA' | 'propagation_delay_ns'
# ==========================================
PARAM_PREFIX = 'iddq_uA'

col_0h = f'{PARAM_PREFIX}_0h'
col_24h = f'{PARAM_PREFIX}_24h'
col_168h = f'{PARAM_PREFIX}_168h'

df = pd.read_csv('clean_burnin_dataset.csv')

# Drop NaNs specific to active parameter features & target
clean_df = df.dropna(subset=[col_0h, col_24h, col_168h]).copy()

# Feature engineering
clean_df['drift_0_to_24'] = clean_df[col_24h] - clean_df[col_0h]
clean_df['ratio_24_to_0'] = clean_df[col_24h] / (clean_df[col_0h] + 1e-6)

X = clean_df[[col_0h, col_24h, 'drift_0_to_24', 'ratio_24_to_0']]
y = clean_df[col_168h]

# Log transformation on target (no data leakage)
y_log = np.log1p(y)

X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)


In [7]:
# ==========================================
# 2. BAYESIAN OPTIMIZATION SEARCH SPACE
# ==========================================
search_space = {
    'n_estimators': Integer(50, 300),
    'max_depth': Integer(3, 8),
    'learning_rate': Real(0.01, 0.2, prior='log-uniform'),
    'subsample': Real(0.6, 1.0)
}

opt = BayesSearchCV(
    estimator=GradientBoostingRegressor(random_state=42),
    search_spaces=search_space,
    n_iter=15,  # Number of Bayesian optimization iterations
    cv=3,
    scoring='neg_mean_squared_error',
    random_state=42,
    n_jobs=-1
)

# Fit Bayesian Search
opt.fit(X_train, y_train_log)

# Best estimator found by Bayesian optimization
best_model = opt.best_estimator_

In [8]:
# ==========================================
# 3. EVALUATION & DRIFT PREDICTION
# ==========================================
y_pred_log = best_model.predict(X_test)
y_pred = np.expm1(y_pred_log) 
y_test = np.expm1(y_test_log)

r2_log = r2_score(y_test_log, y_pred_log)
 

# Metric splits
overall_mae = mean_absolute_error(y_test, y_pred)
normal_mask = y_test < 100
normal_mae = mean_absolute_error(y_test[normal_mask], y_pred[normal_mask])

print(f"=== Bayesian Optimization Results ({PARAM_PREFIX}) ===")
print(f"Best Hyperparameters: {opt.best_params_}")
print(f"Log Scale R^2       : {r2_log:.4f}")
 
print(f"Normal Parts MAE    : {normal_mae:.4f}")
print(f"Overall MAE         : {overall_mae:.4f}")

=== Bayesian Optimization Results (iddq_uA) ===
Best Hyperparameters: OrderedDict({'learning_rate': 0.114012258603381, 'max_depth': 4, 'n_estimators': 200, 'subsample': 0.9211059124625243})
Log Scale R^2       : 0.9906
Normal Parts MAE    : 0.6105
Overall MAE         : 0.6105


In [9]:
'''# ==========================================
# 4. CONSTRUCT & PRINT PREDICTION TABLE
# ==========================================
# Construct DataFrame attached to original test index
results_df = X_test.copy()
results_df['Actual_168h'] = y_test
results_df['Predicted_168h'] = y_pred
results_df['Absolute_Error'] = (results_df['Actual_168h'] - results_df['Predicted_168h']).abs()

# Define table columns to display
table_cols = [col_0h, col_24h, 'Actual_168h', 'Predicted_168h', 'Absolute_Error']

print(f"=== Predictions Table ({PARAM_PREFIX}) ===")
print(results_df[table_cols].head(20).to_string())'''

'# ==========================================\n# 4. CONSTRUCT & PRINT PREDICTION TABLE\n# ==========================================\n# Construct DataFrame attached to original test index\nresults_df = X_test.copy()\nresults_df[\'Actual_168h\'] = y_test\nresults_df[\'Predicted_168h\'] = y_pred\nresults_df[\'Absolute_Error\'] = (results_df[\'Actual_168h\'] - results_df[\'Predicted_168h\']).abs()\n\n# Define table columns to display\ntable_cols = [col_0h, col_24h, \'Actual_168h\', \'Predicted_168h\', \'Absolute_Error\']\n\nprint(f"=== Predictions Table ({PARAM_PREFIX}) ===")\nprint(results_df[table_cols].head(20).to_string())'

In [10]:
# ==========================================
# 4. CONSTRUCT & PRINT PREDICTION TABLE (WITH SLOPE & FLAGGING)
# ==========================================
# Construct DataFrame attached to original test index
results_df = X_test.copy()
results_df['Actual_168h'] = y_test
results_df['Predicted_168h'] = y_pred
results_df['Absolute_Error'] = (results_df['Actual_168h'] - results_df['Predicted_168h']).abs()

# Calculate predicted slope (rate of drift from 24h to predicted 168h per hour)
time_delta_hours = 168 - 24
results_df['Predicted_Slope'] = (results_df['Predicted_168h'] - results_df[col_24h]) / time_delta_hours

# Dynamic outlier threshold using IQR (1.5 * IQR above Q3)
q75 = results_df['Predicted_Slope'].quantile(0.75)
q25 = results_df['Predicted_Slope'].quantile(0.25)
iqr = q75 - q25
slope_threshold = q75 + (1.5 * iqr)

# Flag predictions that exceed the dynamic slope threshold
results_df['High_Slope_Flag'] = results_df['Predicted_Slope'] > slope_threshold

# Define table columns to display
table_cols = [col_0h, col_24h, 'Actual_168h', 'Predicted_168h', 'Absolute_Error', 'Predicted_Slope', 'High_Slope_Flag']

print(f"\nDynamic Slope Cutoff Threshold: {slope_threshold:.6f}")
print(f"=== Predictions Table ({PARAM_PREFIX}) ===")
print(results_df[table_cols].head(20).to_string())


Dynamic Slope Cutoff Threshold: 0.016064
=== Predictions Table (iddq_uA) ===
      iddq_uA_0h  iddq_uA_24h  Actual_168h  Predicted_168h  Absolute_Error  Predicted_Slope  High_Slope_Flag
592       8.6544       8.9950       8.9000        9.216225        0.316225         0.001536            False
4397      7.2204       7.3659       8.4046        7.638649        0.765951         0.001894            False
5794     12.4015      13.2515      13.7503       13.255399        0.494901         0.000027            False
3296      6.8493       7.1121       7.1421        7.326740        0.184640         0.001491            False
5309     16.1637      15.7050      18.3721       17.222395        1.149705         0.010537            False
4139     10.8418      10.6959      11.9791       11.620451        0.358649         0.006420            False
6009     16.8660      18.0930      20.3628       18.381835        1.980965         0.002006            False
3033      5.3782       5.0101       5.7115        